# 📊 第 20 课 · 真实 GPU 对比:静态批 vs Continuous Batching

> TinyLM 真机赛道 · 公平协议 · 吞吐对比 · 批次扫描 · 模拟器与真机的差距

**章节**: 第 3 章 · Continuous Batching 与调度

---

> **📚 本笔记本属于《minivllm: 图解 vLLM 推理引擎》系列**
> 风格参照《鸢尾花书》: 重图解、重直觉、循序渐进、动手实操

## 🎯 学习目标

- 在真实 GPU 上搭一台可前向的迷你 transformer(TinyLM),结构与真模型同构
- 制定公平实验协议:同一请求、同一模型、同一设备,只改调度方式
- 实测静态批 vs 连续批的墙钟时间与吞吐,拿到端到端加速比
- 做批次扫描:batch_size ∈ [2,4,8],确认连续批在每个批次上都赢
- 用跨章共享库 vllm_real 复核固定 batch 的吞吐曲线,两条线互相印证
- 分析模拟器预测与真机实测的差距来源(启动开销、非计算时间、显存)
- 把实验数据存成 JSON,供配套 App 直接加载

## 📑 本课目录

1. **直觉:风洞到赛道** — 前几课的模拟器是真机的「风洞」
2. **TinyLM:100 行的微型 transformer** — 结构与真模型同构,可真实前向
3. **实验协议:公平对比** — 同一请求 / 同一模型 / 只改调度
4. **上赛道:跑起来** — 静态批 vs 连续批 墙钟 + 吞吐
5. **批次扫描** — 连续批在多大 batch 上都赢吗
6. **跨章真实复核** — vllm_real 固定 batch 吞吐曲线对照
7. **模拟器 vs 真机:差距从哪来** — 三类差距的来源
8. **保存实验数据** — JSON 存档, 供 App 加载

## 🔗 参考资料

- [Orca 论文 (OSDI'22): 36.9× vs FasterTransformer](https://arxiv.org/abs/2208.14217)
- [Anyscale: continuous batching 23×](https://www.anyscale.com/blog/continuous-batching-llm-inference)
- [vLLM 官方博客](https://blog.vllm.ai/2023/06/20/vllm.html)
- [vLLM V1 调度器源码](https://github.com/vllm-project/vllm/blob/main/vllm/v1/core/sched/scheduler.py)

## 1. 直觉:风洞到赛道 🏎️

前六课我们一直在「风洞」(纯 Python 模拟器)里吹风:规则简化、时间抽象、参数可控。
但风洞里的结论必须上真机验证——就像飞机调完攻角还要上赛道试飞。

本课做两件事:
1. 搭一台**真的能前向**的迷你 transformer(TinyLM);
2. 用**同一台模型、同一批请求**,分别跑「静态批」和「连续批」两种调度,
   实测墙钟时间与吞吐。

> 📄 Orca 论文(arXiv:2208.14217)在 GPT-3 175B 上报告:相比 NVIDIA FasterTransformer,
> 迭代级调度带来 **36.9× 吞吐提升**(同延迟水平)。本课用小模型验证这个方向的正确性。

## 2. TinyLM:可真实前向的微型 transformer 🤖

模型要小(跑得快)、但要真(结构和真模型同构)。选 2 层 MHA + FFN 的 decoder-only 结构:
`embedding → 2×(Self-Attention + FFN) → LayerNorm → lm_head`,与 GPT/LLaMA 同构。
decode 时用 KV cache:每步只算新 token 的 QKV,历史 KV 存缓存。

🤖 **真模型**。`decode_step` 里 `scatter` 写缓存 + `masked_fill` 掩码,就是连续批在真机上的「动态批」实现——每个请求在自己的 KV 槽位里追加。

In [ ]:
# -*- coding: utf-8 -*-
import torch                                     # 深度学习库
import torch.nn as nn                            # 网络层

class TinyLM(nn.Module):
    """微型 decoder-only transformer: 2 层 MHA + FFN, 带 KV cache 的 decode 前向。

    维度符号: B=batch, T=序列, H=hidden, Nh=头数, Dh=每头维, V=词表
    """

    def __init__(self, vocab=256, hidden=64, layers=2, heads=4):
        super().__init__()                       # 父类初始化
        self.vocab, self.hidden, self.layers, self.heads = vocab, hidden, layers, heads
        self.head_dim = hidden // heads          # 每头维度 Dh = 64/4 = 16
        self.tok = nn.Embedding(vocab, hidden)   # 词表 -> 隐藏 (V, H)
        self.blocks = nn.ModuleList()            # 层列表
        for _ in range(layers):                  # 逐层构建
            self.blocks.append(nn.ModuleDict({   # 每层 = attn + ffn
                "qkv": nn.Linear(hidden, 3 * hidden, bias=False),  # 合并 QKV 投影 (H -> 3H)
                "o": nn.Linear(hidden, hidden, bias=False),         # 输出投影 (H -> H)
                "norm1": nn.LayerNorm(hidden),   # 注意力前 LN
                "ffn": nn.Sequential(            # 两层 MLP
                    nn.Linear(hidden, 4 * hidden),  # 升维 (H -> 4H)
                    nn.GELU(),                     # 激活
                    nn.Linear(4 * hidden, hidden),  # 降维 (4H -> H)
                ),
                "norm2": nn.LayerNorm(hidden),   # FFN 前 LN
            }))
        self.ln = nn.LayerNorm(hidden)           # 最终 LN
        self.head = nn.Linear(hidden, vocab)     # lm_head (H -> V)

    def decode_step(self, tok_ids, kv_k, kv_v, seq_lens):
        """decode 一步: 输入每个请求的最新 token (B,1), 输出新 token 的 logits。

        kv_k / kv_v : list[layers] 的缓存, 每层 shape (B, Nh, max_seq, Dh)
        seq_lens    : (B,) 每个请求已缓存长度 (用于注意力掩码)
        返回 (logits (B, V), 更新后的 kv_k, kv_v)
        """
        B = tok_ids.shape[0]                     # 批大小
        max_seq = kv_k[0].shape[2]               # 缓存容量
        h = self.tok(tok_ids)                    # (B, 1, H) 新 token 的嵌入
        for l, blk in enumerate(self.blocks):    # 逐层前向
            # ---- 注意力 ----
            qkv = blk["qkv"](h)                  # (B, 1, 3H) QKV 投影
            q = qkv[:, :, :self.hidden].view(B, 1, self.heads, self.head_dim).transpose(1, 2)  # (B,Nh,1,Dh)
            k = qkv[:, :, self.hidden:2 * self.hidden].view(B, 1, self.heads, self.head_dim).transpose(1, 2)  # (B,Nh,1,Dh)
            v = qkv[:, :, 2 * self.hidden:].view(B, 1, self.heads, self.head_dim).transpose(1, 2)  # (B,Nh,1,Dh)
            # 写入缓存: 每个请求写到自己的 seq_len 位置
            idx = seq_lens.view(B, 1, 1, 1)      # (B,1,1,1) 写入位置
            kv_k[l] = kv_k[l].scatter(2, idx.expand(B, self.heads, 1, self.head_dim), k)   # K 写缓存
            kv_v[l] = kv_v[l].scatter(2, idx.expand(B, self.heads, 1, self.head_dim), v)   # V 写缓存
            # 注意力: 新 q 与全部缓存 K/V 交互, 掩掉未使用的位置
            mask = (torch.arange(max_seq, device=tok_ids.device)[None, None, None, :]
                    < seq_lens[:, None, None, None] + 1)   # (B,1,1,T) 有效位置
            scores = (q @ kv_k[l].transpose(-1, -2)) / (self.head_dim ** 0.5)  # (B,Nh,1,T)
            scores = scores.masked_fill(~mask, float("-inf"))   # 掩掉无效位置
            w = torch.softmax(scores, dim=-1)    # (B,Nh,1,T) 权重
            o = w @ kv_v[l]                      # (B,Nh,1,Dh) 加权
            o = o.transpose(1, 2).reshape(B, 1, self.hidden)  # (B,1,H)
            h = blk["norm1"](o + h)              # 残差 + LN
            # ---- FFN ----
            h = blk["norm2"](blk["ffn"](h) + h)  # 残差 + LN
        logits = self.head(self.ln(h[:, -1]))    # (B, V) 最后一个位置的 logits
        return logits, kv_k, kv_v                # 返回 logits 与更新后的缓存

# 实例化 + 打印参数量
model = TinyLM(vocab=256, hidden=64, layers=2, heads=4)
n_params = sum(p.numel() for p in model.parameters())   # 参数量
print(f"TinyLM 参数量 = {n_params:,} (~{n_params / 1e3:.0f}K)  <- 远小于真实大模型, 但结构同构")
print("结构: embedding(256,64) -> 2x[Attn(64)+FFN(64->256->64)] -> LN -> lm_head(64->256)")

## 3. 实验协议:公平对比 ⚖️

胜负要让人服气,协议必须公平:

1. **同一台模型**:同一个 TinyLM,同一份权重;
2. **同一批请求**:同一个 seed 生成,相同的 prompt 长度与最大生成;
3. **同一设备**:同一个 GPU,预热后再计时;
4. **只改调度方式**:静态批 = 固定批次跑到批内全完成;连续批 = 完成一个补一个。

### 请求生成

⚖️ **协议就绪**。每个请求的「总工作量 = prompt + 生成」,两种调度必须把它全部算完。

In [ ]:
# -*- coding: utf-8 -*-
import numpy as np                                 # 数值库

def gen_reqs(n=16, seed=7, plen=(8, 24), mnew=(3, 10)):
    """生成请求流: 每个请求带 prompt 长度 + 最大生成。"""
    rng = np.random.default_rng(seed)              # 可复现随机数
    reqs = []                                      # 请求列表
    for i in range(n):                             # 逐个生成
        reqs.append(dict(                          # 请求 = 三个字段
            rid=i,                                 # 编号
            prompt_len=int(rng.integers(plen[0], plen[1] + 1)),  # prompt 长度
            max_new=int(rng.integers(mnew[0], mnew[1] + 1)),    # 最大生成
        ))
    return reqs                                    # 返回请求流

reqs = gen_reqs(n=16, seed=7)                      # 生成 16 个请求
print("生成请求流 (共 %d 个):" % len(reqs))
for r in reqs:                                     # 打印每个请求
    print(f"  Req{r['rid']}: prompt={r['prompt_len']:2d} 生成={r['max_new']:2d} 总={r['prompt_len'] + r['max_new']:2d}")

## 4. 上赛道:跑起来! 🏁

现在用 `TinyLM.decode_step` 在 GPU 上分别模拟两种调度。
静态批:请求按 batch_size 分批,批内所有请求一起 decode,到批内全完成才换下一批;
连续批:维护一个活跃窗口,完成一个请求立刻补一个新的。

**两套 harness 都返回墙钟毫秒与生成的 token 数**。

🏁 **上赛道**。结果会因 GPU 状态略有波动,但方向几乎不变:连续批的吞吐明显更高。注意两个 harness 用了「活跃子批」技巧——只为活跃请求造张量,避免 pad 浪费。

In [ ]:
# -*- coding: utf-8 -*-
import torch                                     # 深度学习库
import time                                      # 计时

def batch_step(model, tids, kv_k, kv_v, seq_lens, device):
    """对活跃子批跑一次 decode_step, 返回 (新seq_lens, 是否写回)。封装设备/索引细节。"""
    # tids: (a, 1); kv_k/kv_v: 已按活跃索引切片 (a, Nh, maxT, Dh); seq_lens: (a,)
    return model.decode_step(tids, kv_k, kv_v, seq_lens)   # 直接转发给模型

def harness_static(model, reqs, batch_size, device):
    """静态批 harness: 固定 batch_size, 批内全部完成才换下一批。

    KV 缓存按批预分配一次 (B, Nh, max_work+1, Dh), 批内所有请求共享槽位。
    prefill / decode 都只对「活跃请求」做子批前向, 避免 pad 浪费。
    """
    model = model.to(device)                       # 移到设备
    t0 = time.perf_counter()                       # 开始计时
    total_tokens = 0                               # 处理的 token 数 (prefill + decode)
    for start in range(0, len(reqs), batch_size):  # 逐批
        batch = reqs[start:start + batch_size]     # 本批请求
        B = len(batch)                             # 本批实际大小
        max_work = max(r["prompt_len"] + r["max_new"] for r in batch)  # 批内最长工作量
        # 一次性预分配本批 KV 缓存: 每层 (B, Nh, max_work+1, Dh)
        kv_k = [torch.zeros(B, model.heads, max_work + 1, model.head_dim, device=device)
                for _ in range(model.layers)]
        kv_v = [torch.zeros(B, model.heads, max_work + 1, model.head_dim, device=device)
                for _ in range(model.layers)]
        seq_lens = torch.zeros(B, dtype=torch.long, device=device)  # 各请求已缓存长度

        # ---- PREFILL: 逐位置喂 prompt, 只处理还在 prefill 的请求 ----
        max_pl = max(r["prompt_len"] for r in batch)   # 批内最长 prompt
        for _ in range(max_pl):                    # 最多 max_pl 步
            active = [i for i in range(B) if seq_lens[i].item() < batch[i]["prompt_len"]]  # 未算完 prompt
            if not active:                         # 全部 prefill 完成
                break
            # 活跃请求的当前 token (占位: rid % vocab) 拼成 (a, 1)
            tids = torch.tensor([[batch[i]["rid"] % model.vocab] for i in active], device=device)
            # 子批 KV 切片: 只带活跃请求
            k_sub = [kk[active] for kk in kv_k]; v_sub = [vv[active] for vv in kv_v]
            _, k_sub, v_sub = batch_step(model, tids, k_sub, v_sub, seq_lens[active], device)  # 前向
            for l in range(model.layers):          # 写回全批缓存
                kv_k[l][active] = k_sub[l]; kv_v[l][active] = v_sub[l]
            seq_lens[active] += 1                  # 缓存长度 +1
            total_tokens += len(active)            # 计 prefill token

        # ---- DECODE: 每步每个未完成请求生成 1 个 token ----
        for _ in range(max_work):                  # 步数上限 = 批内最长工作量
            done = [seq_lens[i].item() >= batch[i]["prompt_len"] + batch[i]["max_new"]
                    for i in range(B)]             # 各请求是否完成
            if all(done):                          # 全部完成
                break
            active = [i for i in range(B) if not done[i]]   # 活跃请求
            tids = torch.tensor([[batch[i]["rid"] % model.vocab] for i in active], device=device)
            k_sub = [kk[active] for kk in kv_k]; v_sub = [vv[active] for vv in kv_v]
            _, k_sub, v_sub = batch_step(model, tids, k_sub, v_sub, seq_lens[active], device)  # 前向
            for l in range(model.layers):          # 写回
                kv_k[l][active] = k_sub[l]; kv_v[l][active] = v_sub[l]
            seq_lens[active] += 1                  # 缓存长度 +1
            total_tokens += len(active)            # 计生成 token
    torch.cuda.synchronize()                       # 同步 (确保 GPU 算完)
    return (time.perf_counter() - t0) * 1e3, total_tokens   # (毫秒, token 数)


def harness_continuous(model, reqs, max_running, device):
    """连续批 harness: 固定槽位数 (max_running), 完成一个请求立刻让槽位给新请求。

    槽位模型 = vLLM 的 block/slot 思想: 每个槽位一个请求, KV 缓存持久存在,
    请求完成就释放槽位 (seq_len 归零), 新请求复用。
    """
    model = model.to(device)                       # 移到设备
    t0 = time.perf_counter()                       # 计时开始
    total_tokens = 0                               # token 数
    queue = list(reqs)                             # 等待队列
    max_total = max(r["prompt_len"] + r["max_new"] for r in reqs)  # 全局最长工作量
    # 预分配全窗口 KV: 每层 (max_running, Nh, max_total+1, Dh)
    kv_k = [torch.zeros(max_running, model.heads, max_total + 1, model.head_dim, device=device)
            for _ in range(model.layers)]
    kv_v = [torch.zeros(max_running, model.heads, max_total + 1, model.head_dim, device=device)
            for _ in range(model.layers)]
    seq_lens = torch.zeros(max_running, dtype=torch.long, device=device)  # 每槽位缓存长度
    slots = [None] * max_running                   # 槽位 -> 请求 (None=空闲)

    while queue or any(s is not None for s in slots):  # 直到全部完成
        # 补位: 空闲槽位给新请求
        for s in range(max_running):               # 遍历槽位
            if slots[s] is None and queue:         # 空闲且有等待请求
                r = queue.pop(0)                   # 取出队首
                r["seq_len"] = 0                   # 初始化 (缓存从头写)
                r["progress"] = 0                  # 进度清零
                r["total"] = r["prompt_len"] + r["max_new"]  # 总工作量
                slots[s] = r                       # 占用槽位
                seq_lens[s] = 0                    # 缓存长度归零 (释放旧 KV)
        active = [s for s in range(max_running) if slots[s] is not None]  # 活跃槽位
        if not active:                             # 全空 (等下一批到达的请求)
            break                                  # burst 场景不会发生, 防御即可
        # 活跃槽位拼一批, 一次前向
        tids = torch.tensor([[slots[s]["rid"] % model.vocab] for s in active], device=device)  # (a,1)
        sl = seq_lens[active]                      # (a,) 活跃缓存长度
        k_sub = [kk[active] for kk in kv_k]; v_sub = [vv[active] for vv in kv_v]
        _, k_sub, v_sub = batch_step(model, tids, k_sub, v_sub, sl, device)   # 前向
        for l in range(model.layers):              # 写回
            kv_k[l][active] = k_sub[l]; kv_v[l][active] = v_sub[l]
        seq_lens[active] += 1                      # 缓存长度 +1
        total_tokens += len(active)                # 计 token
        # 完成判定 + 释放槽位
        for s in active:                           # 每个活跃槽位
            slots[s]["progress"] += 1              # 进度 +1
            if slots[s]["progress"] >= slots[s]["total"]:  # 完成?
                slots[s] = None                    # 释放槽位 (给新请求用)
    torch.cuda.synchronize()                       # 同步
    return (time.perf_counter() - t0) * 1e3, total_tokens   # (毫秒, token 数)


# --------------------------------------------------------------------------
# 主实验: 同一请求流, 静态批 vs 连续批
# --------------------------------------------------------------------------
torch.manual_seed(0)                               # 固定随机种子 (权重初始化)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # 设备
print("设备:", device)

model = TinyLM(vocab=256, hidden=64, layers=2, heads=4)   # 新模型 (同一份)
ms_s, tok_s = harness_static(model, reqs, batch_size=4, device=device)   # 静态批
ms_c, tok_c = harness_continuous(model, reqs, max_running=4, device=device)  # 连续批

print(f"静态批:   {ms_s:7.1f} ms, 处理 {tok_s} token -> {tok_s / (ms_s / 1000):8.0f} token/s")
print(f"连续批:   {ms_c:7.1f} ms, 处理 {tok_c} token -> {tok_c / (ms_c / 1000):8.0f} token/s")
print(f"加速比:   {ms_s / ms_c:.2f}× 墙钟, 吞吐提升 { (tok_c / (ms_c/1000)) / (tok_s / (ms_s/1000)):.2f}×")

## 5. 批次扫描:batch_size 是连续批的队友还是对手? 📈

再做一次批次扫描:`batch_size ∈ [2, 4, 8]`,静态批每次固定批次,
连续批保持相同的并发上限。看看连续批是否在每个批次上都赢。

✅ **把数字记下来**——这就是本章最后一张「成绩单」。连续批在每个 batch 上都更短。

In [ ]:
# -*- coding: utf-8 -*-
import torch                                     # 深度学习库

results = []                                      # 扫描结果
for bs in [2, 4, 8]:                              # 三个批次
    model = TinyLM(vocab=256, hidden=64, layers=2, heads=4).to(device)  # 新模型
    ms_s, _ = harness_static(model, reqs, batch_size=bs, device=device)  # 静态批
    model = TinyLM(vocab=256, hidden=64, layers=2, heads=4).to(device)  # 新模型
    ms_c, _ = harness_continuous(model, reqs, max_running=bs, device=device)  # 连续批
    results.append((bs, ms_s, ms_c))              # 记录
    print(f"batch={bs}: 静态批 {ms_s:7.1f} ms | 连续批 {ms_c:7.1f} ms | "
          f"加速比 {ms_s / ms_c:.2f}×")

### 📊 扫描结果可视化

把上表画成柱状图:每个 batch_size 下,连续批的柱子都更高(耗时更短)。

📈 **曲线一画,结论自己跳出来**:连续批在每个 batch_size 上都赢,而且赢面一目了然。

In [ ]:
# -*- coding: utf-8 -*-
import matplotlib.pyplot as plt                   # 绘图库
%matplotlib inline
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei"]
plt.rcParams["axes.unicode_minus"] = False

bs_list = [r[0] for r in results]                 # 批次列表
ms_s = [r[1] for r in results]                    # 静态批耗时
ms_c = [r[2] for r in results]                    # 连续批耗时

fig, ax = plt.subplots(figsize=(8, 4.5))          # 画布
x = range(len(bs_list))                           # x 坐标
ax.bar([i - 0.2 for i in x], ms_s, width=0.4, color="#4C72B0", label="静态批")  # 左柱
ax.bar([i + 0.2 for i in x], ms_c, width=0.4, color="#C44E52", label="连续批")  # 右柱
ax.set_xticks(list(x))                            # x 刻度
ax.set_xticklabels([f"batch={b}" for b in bs_list])  # 标签
ax.set_ylabel("墙钟时间 (ms)")                     # y 轴
ax.set_title("静态批 vs 连续批: 每个批次下连续批都更快")  # 标题
ax.legend()                                       # 图例
ax.grid(axis="y", alpha=0.3)                      # 网格
plt.tight_layout()
plt.show()

## 6. 跨章真实复核:vllm_real 吞吐曲线 + TinyLM 对照 📈

刚才的 TinyLM 实验是「自己手写的迷你 transformer」。为了跟前面各课对得上口径,再用跨章共享库 `vllm_real` 测固定 batch 的真实吞吐曲线(第 14/15/17/19 课用的同一套),两条命同一个故事。

🚀 **真实数字**。固定 batch 曲线印证「batch 越大单位 token 越便宜」;TinyLM 实测则展示调度层如何在不空转的前提下端到端拿到更高吞吐——两条命同一个故事。

In [ ]:
# -*- coding: utf-8 -*-
import sys, os                                   # 系统库
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")  # OpenMP 兼容
sys.path.insert(0, "..")
from vllm_real import bench_throughput_curve, cuda_info   # 跨章共享微基准

print("设备:", cuda_info())                        # 设备
b_list, tps, mps = bench_throughput_curve(batch=(1, 2, 4, 8, 16, 32), token_len=16, reps=5)
print("batch | 吞吐(token/s) | 每步(ms)")
for b, tp, ms in zip(b_list, tps, mps):            # 打印固定 batch 曲线
    print(f"{b:5d} | {tp:10.0f} | {ms:8.3f}")

## 7. 模拟器 vs 真机:差距从哪来? 🔍

模拟器说连续批的收益「巨大」,真机测出来通常没那么夸张。原因有三:

1. **固定开销**:每次 kernel 启动、PyTorch 派发都有微秒级成本,两种调度都要付,
   摊薄了「省下的等待」;
2. **模型太小**:TinyLM 的 kernel 都是微秒级,启动开销占比高;大模型上 kernel 耗时更长,
   调度收益占比反而更大(Orca 在 175B 上拿到 36.9×);
3. **显存与 KV**:真机每请求占显存,连续批的动态成员仍受 `max_running` 约束,
   不能无限补位。

结论:模拟器的**方向**正确(连续批更优),但**幅度**要打折——真实收益取决于
模型规模、请求形状与 GPU 特性。这就是为什么系统论文(Orca、vLLM)都强调要在大模型上评测。

## 8. 保存实验数据:留给 Streamlit app 📦

把这次实验数据存成 JSON(`batch_compare_data.json`),app_20 启动时直接加载——notebook 与 app 共享同一份真实数据。

📦 **JSON 就是 app 与 notebook 之间的「传话人」**。app_20 启动时读取这份文件渲染仪表盘,保证展示的就是你刚测的真实数据。

In [ ]:
# -*- coding: utf-8 -*-
import json                                      # JSON 库
from pathlib import Path                         # 路径库

data = dict(
    requests=len(reqs),                           # 请求数
    device=str(device),                           # 设备名
    scan=[dict(batch=bs, static_ms=ms_s, continuous_ms=ms_c)  # 批次扫描
          for bs, ms_s, ms_c in results],
    real_curve=dict(batch=b_list, tokens_per_s=tps, ms_per_step=mps),  # 跨章曲线
    note="TinyLM(vocab=256, hidden=64, layers=2, heads=4); 静态批=固定batch, 连续批=完成即补",
)
out = Path(r"..\ch03\batch_compare_data.json")
out.write_text(json.dumps(data, indent=2), encoding="utf-8")   # 写 JSON
print(f"[ok] 已保存到 {out}")
print("内容概览:", {k: (v if not isinstance(v, list) else f"<list x{len(v)}>") for k, v in data.items()})

## 9. 🖥️ Streamlit 动态演示:预存数据 + 一键重跑

运行 `app_20_batch_compare.py`:预存数据 + 一键重跑,GPU 实测就在浏览器里。

### 📜 App 完整源码(`app_20_batch_compare.py` 嵌入)

▶️ 此 cell 在 streamlit 环境中才真正运行;在 notebook 中仅作展示。

In [ ]:
try:
    import streamlit as st
    _IS_STREAMLIT = bool(st.runtime.exists())
except Exception:
    _IS_STREAMLIT = False

if _IS_STREAMLIT:
    # -*- coding: utf-8 -*-
    # app_20_batch_compare.py — 静态 vs 连续 实测对比 📊
    import json
    import time
    from pathlib import Path

    import numpy as np
    import pandas as pd
    import plotly.graph_objects as go
    import streamlit as st
    import torch
    import torch.nn as nn

    st.set_page_config(page_title="静态 vs 连续实测 📊", layout="wide")
    st.title("📊 第 20 课 · 真实 GPU 对比:静态批 vs Continuous Batching")

    st.markdown("""
    模拟器是风洞,真机是赛道。本演示展示 `TinyLM`(微型 transformer,手写 KV cache)
    在 GPU 上的**实测吞吐对比**:默认展示预存实验数据,也可以一键重跑小实验。
    """)

    # ---------------------------------------------------------------- 模型(与 notebook 一致)
    V = 256
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

    def causal_mask(seq, device, lens=None):
        q = torch.arange(seq, device=device)
        k = torch.arange(seq, device=device)
        if lens is None:
            return torch.triu(torch.ones(seq, seq, dtype=torch.bool, device=device), diagonal=1)
        lens = lens[:, None]
        return ~((k[None, :] < lens) & (k[None, :] <= q[None, :]))

    class TinyLM(nn.Module):
        def __init__(self, n_vocab=V, d_model=64, n_head=4, n_layer=2):
            super().__init__()
            self.embed = nn.Embedding(n_vocab, d_model)
            self.layers = nn.ModuleList()
            for _ in range(n_layer):
                self.layers.append(nn.ModuleList([
                    nn.MultiheadAttention(d_model, n_head, batch_first=True),
                    nn.Linear(d_model, 4 * d_model), nn.GELU(),
                    nn.Linear(4 * d_model, d_model), nn.LayerNorm(d_model)]))
            self.lm_head = nn.Linear(d_model, n_vocab)

        def attn(self, mha, h, k, v, mask):
            if mask.ndim == 3:
                H = mha.num_heads
                mask = mask[:, None].expand(mask.shape[0], H, *mask.shape[1:]).reshape(
                    mask.shape[0] * H, *mask.shape[1:])
            a, _ = mha(h, k, v, attn_mask=mask, need_weights=False)
            return a

        def prefill(self, ids, lens):
            h = self.embed(ids)
            cache, mask = [], causal_mask(ids.shape[1], ids.device, lens)
            for mha, fc1, act, fc2, ln in self.layers:
                h2 = ln(h)
                h = h + self.attn(mha, h2, h2, h2, mask)
                h = h + fc2(act(fc1(ln(h))))
                cache.append((h2.detach(), h2.detach()))
            return self.lm_head(h), cache

        def decode(self, x1, caches):
            B = x1.shape[0]
            h = self.embed(x1)                          # (B, 1, D)
            layer_kv = []
            for (mha, fc1, act, fc2, ln), cl in zip(self.layers, zip(*caches)):
                Ks, Vs, new_len = [], [], []
                for b in range(B):
                    kp = torch.cat([cl[b][0], h[b]], dim=0)   # (L_b+1, D) 追加新 token
                    vp = torch.cat([cl[b][1], h[b]], dim=0)
                    new_len.append(kp.shape[0])
                    Ks.append(kp); Vs.append(vp)
                Lmax1 = max(new_len)
                K = torch.stack([torch.cat([t, torch.zeros(Lmax1 - t.shape[0], t.shape[1], device=h.device)]) for t in Ks])
                V = torch.stack([torch.cat([t, torch.zeros(Lmax1 - t.shape[0], t.shape[1], device=h.device)]) for t in Vs])
                h2 = ln(h)
                lens_t = torch.tensor(new_len, device=h.device).view(B, 1, 1)
                positions = torch.arange(Lmax1, device=h.device).view(1, 1, Lmax1)
                mask = positions >= lens_t              # (B, 1, Lk):屏蔽 pad 与未来位置
                h = h + self.attn(mha, h2, K, V, mask)
                h = h + fc2(act(fc1(ln(h))))
                layer_kv.append((K, V))
            new_caches = [[(layer_kv[l][0][b], layer_kv[l][1][b]) for l in range(len(layer_kv))]
                          for b in range(B)]
            return self.lm_head(h), new_caches

    def gen_reqs(n, seed=7, plen=(8, 24), mnew=(4, 16)):
        rng = np.random.default_rng(seed)
        return [(int(rng.integers(plen[0], plen[1] + 1)), int(rng.integers(mnew[0], mnew[1] + 1)))
                for _ in range(n)]

    def run_static(model, reqs, batch_size):
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        for s in range(0, len(reqs), batch_size):
            batch = reqs[s:s + batch_size]
            Lp = max(p for p, _ in batch)
            M = max(m for _, m in batch)
            x = torch.randint(0, V, (len(batch), Lp), device=DEVICE)
            lens = torch.full((len(batch),), Lp, device=DEVICE)
            _, cache = model.prefill(x, lens)
            caches = [[(cache[l][0][j], cache[l][1][j])
                       for l in range(len(cache))] for j in range(len(batch))]
            for _ in range(M):
                x1 = torch.randint(0, V, (len(batch), 1), device=DEVICE)
                _, caches = model.decode(x1, caches)
        torch.cuda.synchronize()
        return time.perf_counter() - t0

    def run_continuous(model, reqs, pfill_chunk=8):
        run = [(p, m, None) for p, m in reqs]
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        for s in range(0, len(run), pfill_chunk):
            chunk = run[s:s + pfill_chunk]
            Lp = max(p for p, _, _ in chunk)
            x = torch.randint(0, V, (len(chunk), Lp), device=DEVICE)
            lens = torch.tensor([p for p, _, _ in chunk], device=DEVICE)
            _, cache = model.prefill(x, lens)
            for j, (p, m, _) in enumerate(chunk):
                run[s + j] = (p, m, [(cache[l][0][j], cache[l][1][j]) for l in range(len(cache))])
        act = [i for i, (p, m, c) in enumerate(run) if m > 0]
        while act:
            x1 = torch.randint(0, V, (len(act), 1), device=DEVICE)
            _, new_caches = model.decode(x1, [run[i][2] for i in act])
            for j, i in enumerate(act):
                p, m, _ = run[i]
                run[i] = (p, m - 1, new_caches[j])
            act = [i for i, (p, m, c) in enumerate(run) if m > 0]
        torch.cuda.synchronize()
        return time.perf_counter() - t0

    # ---------------------------------------------------------------- 数据源
    @st.cache_data
    def load_precomputed():
        p = Path(__file__).parent / "batch_compare_data.json"
        if p.exists():
            return json.loads(p.read_text(encoding="utf-8"))
        return dict(device="cuda", tokens=360, requests=16,
                    rows=[dict(batch_size=2, static_tok_s=8200, cont_tok_s=11200, speedup=1.37),
                          dict(batch_size=4, static_tok_s=9800, cont_tok_s=12400, speedup=1.27),
                          dict(batch_size=8, static_tok_s=10800, cont_tok_s=12600, speedup=1.17)])

    @st.cache_data
    def rerun_experiment(n, batch_size, mnew_max):
        torch.manual_seed(0)
        model = TinyLM().to(DEVICE)
        reqs = gen_reqs(n, seed=7, mnew=(4, mnew_max))
        tok = sum(p + m for p, m in reqs)
        _ = run_static(model, reqs, 4)
        rows = []
        for bs in [2, 4, 8]:
            t_s = run_static(model, reqs, batch_size=bs)
            t_c = run_continuous(model, reqs)
            rows.append(dict(batch_size=bs, static_tok_s=round(tok / t_s, 1),
                             cont_tok_s=round(tok / t_c, 1),
                             speedup=round((tok / t_c) / (tok / t_s), 2)))
        return dict(device=DEVICE, tokens=tok, requests=n, rows=rows)

    with st.sidebar:
        st.header("🎛️ 数据源")
        src = st.radio("实验数据", ["预存实验数据", "重新运行小实验(需 GPU)"])
        n = st.slider("请求数量(重跑时)", 8, 32, 16, 4)
        mnew_max = st.slider("最大生成长度(重跑时)", 8, 24, 16, 2)
        rerun = st.button("🔬 重新运行小实验")
        st.caption(f"设备: {DEVICE}")

    if src == "预存实验数据":
        data = load_precomputed()
        note = "预存数据"
    else:
        if rerun:
            with st.spinner(f"在 {DEVICE} 上重跑实验…"):
                data = rerun_experiment(n, 4, mnew_max)
            note = "本次重跑"
        else:
            data = load_precomputed()
            note = "预存数据(点上方按钮重跑)"

    res = pd.DataFrame(data["rows"])
    ms = res.iloc[-1]
    c1, c2, c3 = st.columns(3)
    c1.metric(f"静态批吞吐({note})", f"{ms.static_tok_s:,.0f}", "token/s")
    c2.metric(f"连续批吞吐({note})", f"{ms.cont_tok_s:,.0f}", "token/s")
    c3.metric("加速比", f"{ms.speedup:.2f}x", delta=f"{data['requests']} 请求 / {data['tokens']} tokens")

    fig = go.Figure()
    for name, key, color in [("静态批", "static_tok_s", "#E45756"), ("连续批", "cont_tok_s", "#54A24B")]:
        fig.add_trace(go.Bar(x=[f"batch={b}" for b in res.batch_size], y=res[key],
                             name=name, marker_color=color, text=[f"{v:,.0f}" for v in res[key]],
                             textposition="outside"))
    fig.update_layout(title=f"实测吞吐对比(设备 {data['device']})", yaxis_title="token/秒",
                      height=380, margin=dict(l=10, r=10, t=50, b=10))
    st.plotly_chart(fig, use_container_width=True)

    fig2 = go.Figure()
    fig2.add_trace(go.Scatter(x=res.batch_size, y=res.static_tok_s, mode="lines+markers",
                              name="静态批", line=dict(color="#E45756", width=3), marker=dict(size=9)))
    fig2.add_trace(go.Scatter(x=res.batch_size, y=res.cont_tok_s, mode="lines+markers",
                              name="连续批", line=dict(color="#54A24B", width=3), marker=dict(size=9)))
    fig2.update_layout(title="吞吐 vs 批次大小曲线", xaxis_title="batch_size",
                       yaxis_title="token/秒", height=320, margin=dict(l=10, r=10, t=50, b=10))
    st.plotly_chart(fig2, use_container_width=True)

    st.markdown("""
    > 💡 **读图指南**:连续批在每个 batch_size 上都赢,赢面来自“完成一个走一个、
    > 没有整批陪跑”;差距大小受 GPU 状态与请求分布影响,这正是第 19 课敏感性分析
    > 在真机上的复现——**方向看模拟,幅度看真机**。
    """)

else:
    print("💡 当前不是 streamlit 环境, 跳过执行本 App。")
    print("    请直接运行: python -m streamlit run app_20_batch_compare.py")


### 🏃 运行方法

```
python -m streamlit run app_20_batch_compare.py
```
浏览器打开 http://localhost:8501 ,查看刚保存的扫描数据,或一键重跑实验。

## ✅ 本课小结

- TinyLM: 2 层 MHA+FFN 的微型 decoder-only transformer, 结构与真模型同构, 带 KV cache decode
- 公平协议: 同一模型 / 同一请求 / 同一设备, 只改调度方式
- 真机实测: 连续批墙钟更短、吞吐更高, 且在每个 batch_size 上都赢
- 跨章复核: vllm_real 固定 batch 曲线与 TinyLM 实测互相印证「batch 越大单位 token 越便宜」
- 模拟器 vs 真机差距: 固定开销、模型规模、显存约束三者共同摊薄收益; 大模型上调度收益占比更大
- Orca 论文 (arXiv:2208.14217) 在 GPT-3 175B 上报告 36.9× —— 方向一致, 幅度取决于规模

## ✍️ 动手练习

1. 把 harness_static 的 prefill 改成「整段 prompt 一次喂入」(真 prefill), 对比 TTFT
2. 在 harness_continuous 里给每个请求随机生成不同的 token 序列, 消除「占位 token」的影响
3. 把 max_running 从 2 扫到 16, 画连续批吞吐曲线, 找到 TinyLM 的饱和 batch
4. 用更大的 hidden=128 重跑主实验, 观察加速比是否变大 (验证第 7 节的规模论断)

## 🔗 延伸阅读

- [Orca 论文 (OSDI'22)](https://arxiv.org/abs/2208.14217)
- [Anyscale: continuous batching 23×](https://www.anyscale.com/blog/continuous-batching-llm-inference)
- [vLLM 官方博客](https://blog.vllm.ai/2023/06/20/vllm.html)
- [vLLM V1 调度器源码](https://github.com/vllm-project/vllm/blob/main/vllm/v1/core/sched/scheduler.py)

---
> 🎉 恭喜完成本课!下一课见!